# 01 · Data Download & Ingestion

## 0 · Configuration

In [ ]:
# ── S3 ────────────────────────────────────────────────────────────────────────
S3_BUCKET      = "se-data-dump-2025"
S3_PREFIX      = "raw/stackoverflow"
PARQUET_PREFIX = "parquet/stackoverflow"

# ── AWS Network (EC2Cluster) ──────────────────────────────────────────────────
AWS_VPC = 'vpc-016ed068e4f82912e'
AWS_SUBNET_ID = 'subnet-06773955a1f25bb73'
AWS_SECURITY_GROUPS = ['sg-0e2b32f7f7eb7c7f6']

INSTANCE_TYPE        = "r5.xlarge"
N_WORKERS            = 4
IAM_INSTANCE_PROFILE = "LabInstanceProfile"

print(f"S3 raw    : s3://{S3_BUCKET}/{S3_PREFIX}/")
print(f"S3 parquet: s3://{S3_BUCKET}/{PARQUET_PREFIX}/")

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1779593861358_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

S3 raw    : s3://se-data-dump-2025/raw/stackoverflow/
S3 parquet: s3://se-data-dump-2025/parquet/stackoverflow/

---
## Phase A 

Download XML files through SSH terminal and save to `s3://se-data-dump-2025/raw/stackoverflow/`：
- Posts.xml、Users.xml、Votes.xml、PostLinks.xml、Tags.xml

---
## Phase B: XML to Parquet

### B-1 · Imports & AWS Credentials

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

print("Imports OK.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Imports OK.

### B-2 · PySpark set up

In [ ]:
print("Spark version:", spark.version)
print("S3 raw    :", f"s3://{S3_BUCKET}/{S3_PREFIX}/")
print("S3 parquet:", f"s3://{S3_BUCKET}/{PARQUET_PREFIX}/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Spark version: 3.5.6-amzn-2
S3 raw    : s3://se-data-dump-2025/raw/stackoverflow/
S3 parquet: s3://se-data-dump-2025/parquet/stackoverflow/

### B-3 · XML to Parquet

In [ ]:
XML_CONFIGS = {
    "Tags.xml": {
        "cols": ["Id", "TagName", "Count", "ExcerptPostId", "WikiPostId"],
        "date_col": None,
    },
    "Posts.xml": {
        "cols": ["Id", "PostTypeId", "CreationDate", "Score", "Body",
                 "Tags", "OwnerUserId", "AnswerCount", "ClosedDate", "AcceptedAnswerId"],
        "date_col": "CreationDate",
    },
    "Users.xml": {
        "cols": ["Id", "Reputation", "CreationDate", "LastAccessDate"],
        "date_col": "CreationDate",
    },
    "PostLinks.xml": {
        "cols": ["PostId", "RelatedPostId", "LinkTypeId"],
        "date_col": None,
    },
}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

def parse_xml_row(line, target_cols):
    import xml.etree.ElementTree as ET
    line = line.strip()
    if not line.startswith('<row'):
        return None
    try:
        if not line.endswith('/>'):
            line = line + '/>'
        elem = ET.fromstring(line)
        return tuple(elem.attrib.get(c) for c in target_cols)
    except Exception:
        return None

print("Helpers defined.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Helpers defined.

In [ ]:
import time as _time
from pyspark.sql.functions import col, date_format

# ── test pipe line using Tags.xml─────────────────────────────
test_config = {"Tags.xml": XML_CONFIGS["Tags.xml"]}

for xml_name, cfg in test_config.items():
    cols     = cfg["cols"]
    date_col = cfg["date_col"]
    src = f"s3://{S3_BUCKET}/{S3_PREFIX}/{xml_name}"
    dst = f"s3://{S3_BUCKET}/{PARQUET_PREFIX}/{xml_name.replace('.xml', '')}"

    print(f"[TEST] {xml_name}  →  {dst}")
    t0 = _time.time()

    schema = StructType([StructField(c, StringType(), True) for c in cols])
    cols_bc = spark.sparkContext.broadcast(cols)

    rdd = (
        spark.sparkContext.textFile(src)
        .map(lambda line: parse_xml_row(line, cols_bc.value))
        .filter(lambda x: x is not None)
    )
    df = spark.createDataFrame(rdd, schema=schema)

    if xml_name == "Posts.xml":
        df = df.filter(
            (col("PostTypeId") == "1") &
            (
                col("Tags").contains("<python>") |
                col("Tags").contains("<python-3") |
                col("Tags").contains("<python-2")
            )
        )

    if date_col:
        df = df.withColumn("year_month", date_format(col(date_col).cast("timestamp"), "yyyy-MM"))
        df.write.mode("overwrite").partitionBy("year_month").parquet(dst)
    else:
        df.write.mode("overwrite").parquet(dst)

    n = spark.read.parquet(dst).count()
    elapsed = (_time.time() - t0) / 60
    print(f"[TEST OK] rows={n:,}  ({elapsed:.1f} min)")

print("\n验证通过后，运行下方'全量转换' cell。Tags.xml 不会重复跑。")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[TEST] Tags.xml  ?  s3://se-data-dump-2025/parquet/stackoverflow/Tags
[TEST OK] rows=66,137  (0.1 min)

??????????'????' cell?Tags.xml ??????

In [ ]:
# Run full pipeline
import sys
import time as _time
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, date_format

FULL_CONFIGS = {k: v for k, v in XML_CONFIGS.items() if k != "Tags.xml"}

for xml_name, cfg in FULL_CONFIGS.items():
    cols     = cfg["cols"]
    date_col = cfg["date_col"]
    src = f"s3://{S3_BUCKET}/{S3_PREFIX}/{xml_name}"
    dst = f"s3://{S3_BUCKET}/{PARQUET_PREFIX}/{xml_name.replace('.xml', '')}"

    print(f"\n{'='*60}")
    print(f"[{xml_name}]  {src}  →  {dst}")
    t0 = _time.time()

    # ── 进度：Step 1 ──────────────────────────────────────
    print(f"  [1/4] 读取 XML 文本行...", flush=True)
    schema = StructType([StructField(c, StringType(), True) for c in cols])
    cols_bc = spark.sparkContext.broadcast(cols)

    rdd = (
        spark.sparkContext.textFile(src)
        .map(lambda line: parse_xml_row(line, cols_bc.value))
        .filter(lambda x: x is not None)
    )
    print(f"  [1/4] RDD 定义完成（惰性求值，实际解析在写入时触发）", flush=True)

    # ── 进度：Step 2 ──────────────────────────────────────
    print(f"  [2/4] 创建 DataFrame...", flush=True)
    df = spark.createDataFrame(rdd, schema=schema)

    # ── 进度：Step 3 ──────────────────────────────────────
    if xml_name == "Posts.xml":
        print(f"  [3/4] 过滤 Python 相关问题...", flush=True)
        df = df.filter(
            (col("PostTypeId") == "1") &
            (
                col("Tags").contains("<python>") |
                col("Tags").contains("<python-3") |
                col("Tags").contains("<python-2")
            )
        )
    else:
        print(f"  [3/4] 跳过过滤（非 Posts 表）", flush=True)

    # ── 进度：Step 4 ──────────────────────────────────────
    print(f"  [4/4] 写入 Parquet → {dst}", flush=True)
    print(f"        （这一步耗时最长，Posts.xml 约 20–40 分钟）", flush=True)

    write_t0 = _time.time()
    if date_col:
        df = df.withColumn(
            "year_month",
            date_format(col(date_col).cast("timestamp"), "yyyy-MM")
        )
        df.write.mode("overwrite").partitionBy("year_month").parquet(dst)
    else:
        df.write.mode("overwrite").parquet(dst)

    write_elapsed = (_time.time() - write_t0) / 60
    elapsed = (_time.time() - t0) / 60

    # ── 写完后立刻做行数验证 ──────────────────────────────
    print(f"  ✓ 写入完成（写入耗时 {write_elapsed:.1f} min）", flush=True)
    print(f"  正在统计行数...", flush=True)
    n_rows = spark.read.parquet(dst).count()
    print(f"  ✓ 行数：{n_rows:,}", flush=True)
    print(f"[{xml_name}] 全部完成，总耗时 {elapsed:.1f} min")

print("\n" + "="*60)
print("All tables written to Parquet.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


[Posts.xml]  s3://se-data-dump-2025/raw/stackoverflow/Posts.xml  ?  s3://se-data-dump-2025/parquet/stackoverflow/Posts
  [1/4] ?? XML ???...
  [1/4] RDD ?????????????????????
  [2/4] ?? DataFrame...
  [3/4] ?? Python ????...
  [4/4] ?? Parquet ? s3://se-data-dump-2025/parquet/stackoverflow/Posts
        ?????????Posts.xml ? 20?40 ???
  ? ????????? 24.3 min?
  ??????...
  ? ???2,388,184
[Posts.xml] ???????? 24.3 min

[Users.xml]  s3://se-data-dump-2025/raw/stackoverflow/Users.xml  ?  s3://se-data-dump-2025/parquet/stackoverflow/Users
  [1/4] ?? XML ???...
  [1/4] RDD ?????????????????????
  [2/4] ?? DataFrame...
  [3/4] ?????? Posts ??
  [4/4] ?? Parquet ? s3://se-data-dump-2025/parquet/stackoverflow/Users
        ?????????Posts.xml ? 20?40 ???
  ? ????????? 6.2 min?
  ??????...
  ? ???28,628,231
[Users.xml] ???????? 6.2 min

[PostLinks.xml]  s3://se-data-dump-2025/raw/stackoverflow/PostLinks.xml  ?  s3://se-data-dump-2025/parquet/stackoverflow/PostLinks
  [1/4] ?? XML ???...
  [1/4] R

### B-4 · Sanity Checks

In [ ]:
print(f"{'Table':<12} {'Row count':>15}")
print("-" * 30)
for xml_name in XML_CONFIGS:
    table = xml_name.replace('.xml', '')
    path  = f"s3://{S3_BUCKET}/{PARQUET_PREFIX}/{table}"
    n = spark.read.parquet(path).count()
    print(f"{table:<12} {n:>15,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Table              Row count
------------------------------
Tags                  66,137
Posts              2,388,184
Users             28,628,231
PostLinks          6,532,134

In [ ]:
from pyspark.sql.functions import col, count

posts_pq = spark.read.parquet(f"s3://{S3_BUCKET}/{PARQUET_PREFIX}/Posts")

monthly = (
    posts_pq
    .filter(
        (col("PostTypeId") == "1") &
        (col("year_month") >= "2019-01") &
        (col("year_month") <= "2025-03")
    )
    .groupBy("year_month")
    .agg(count("Id").alias("n_questions"))
    .orderBy("year_month")
    .toPandas()
)

monthly["year_month"] = pd.to_datetime(monthly["year_month"])
monthly = monthly.sort_values("year_month")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly["year_month"], monthly["n_questions"],
        linewidth=1.5, color="steelblue")
ax.axvline(pd.Timestamp("2022-11-30"), color="red", linestyle="--",
           linewidth=1.2, label="ChatGPT launch (2022-11-30)")
ax.set_title("Stack Overflow – Monthly Question Volume (2019-01 to 2025-03)")
ax.set_xlabel("Month")
ax.set_ylabel("Questions")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend()
plt.tight_layout()
plt.savefig("monthly_question_volume.png", dpi=150)
plt.show()
print("Plot saved to monthly_question_volume.png")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Plot saved to monthly_question_volume.png

In [ ]:
import boto3
s3 = boto3.client("s3")
s3.upload_file(
    "monthly_question_volume.png",
    S3_BUCKET,
    "outputs/monthly_question_volume.png"
)
print("上传完成")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

????

In [ ]:
posts_pq = spark.read.parquet(f"s3://{S3_BUCKET}/{PARQUET_PREFIX}/Posts")

from pyspark.sql.functions import min, max
posts_pq.select(
    min("year_month").alias("earliest"),
    max("year_month").alias("latest"),
).show()

from pyspark.sql.functions import col
posts_pq.groupBy(col("year_month").substr(1,4).alias("year")) \
    .count() \
    .orderBy("year") \
    .show(20)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+-------+
|earliest| latest|
+--------+-------+
| 2008-08|2025-03|
+--------+-------+

+----+------+
|year| count|
+----+------+
|2008|  2049|
|2009| 12869|
|2010| 27034|
|2011| 42267|
|2012| 65321|
|2013| 99927|
|2014|121299|
|2015|144763|
|2016|170726|
|2017|210306|
|2018|228279|
|2019|254596|
|2020|311862|
|2021|268233|
|2022|237385|
|2023|120146|
|2024| 63183|
|2025|  7939|
+----+------+